# Lista 1 - Wstęp do uczenia maszynowego

## Zadanie 1

Instalacja i konfiguracja środowiska: Utwórz wirtualne środowisko dla kursu. Zainstaluj niezbędne biblioteki: pandas, numpy, scikit-learn, matplotlib.

## Zadanie 2
Wczytanie i eksploracja danych: Wczytaj publicznie dostępny zbiór danych Iris z biblioteki scikit-learn lub bezpośrednio z adresu URL. Wyświetl pierwsze kilka wierszy i podstawowe informacje o zbiorze (`df.info()`, `df.describe()`).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = load_iris(as_frame=True).frame
display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

## Zadanie 3
Obsługa brakujących wartości: Wczytaj zbiór danych Titanic (dostępny na Kaggle). Sprawdź, które kolumny zawierają brakujące wartości. Zaproponuj i zaimplementuj dwie różne strategie ich uzupełniania.

In [ ]:
gender_submission = pd.read_csv('titanic/gender_submission.csv')
train = pd.read_csv('titanic/train.csv')
test = pd.read_csv('titanic/test.csv')

display(gender_submission.head())
display(train.head())
display(test.head())

In [ ]:
print('gender submission shape: ', gender_submission.shape)
print('train shape: ', train.shape)
print('test shape: ', test.shape)

In [ ]:
# Sprawdzenie brakujących wartości w zbiorach danych
print(gender_submission.isna().sum())

In [ ]:
print(train.isna().sum())

In [ ]:
print(test.isna().sum())

### Strategia uzupełniania braków danych

1. Uzupełnianie braków danych najczęściej występującą wartością (modą) - dla kolumn kategorycznych.
2. Uzupełnianie braków danych średnią arytmetyczną lub medianą - dla kolumn numerycznych.
3. Uzupełnianie braków danych wartością stałą (np. "Undefined" lub 0).
4. Pomijanie wierszy z brakami danych.
5. Pomijanie kolumn z brakami danych.

In [ ]:
def fill_na_with_mode(dataframe, columns):
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        most_common = dataframe[column].mode()[0]           # Uzupełnianie braków danych najczęściej występującą wartością
        dataframe[column] = dataframe[column].fillna(most_common)
    return dataframe


def fill_na_with_mean(dataframe, columns):
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        try:
            mean_value = dataframe[column].mean()           # Uzupełnianie braków danych średnią arytmetyczną
            dataframe[column] = dataframe[column].fillna(mean_value)
        except ValueError:
            print(f'Column {column} is not numeric, cannot fill with mean.')
    return dataframe


def fill_na_with_median(dataframe, columns):
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        try:
            median_value = dataframe[column].median()       # Uzupełnianie braków danych medianą
            dataframe[column] = dataframe[column].fillna(median_value)
        except ValueError:
            print(f'Column {column} is not numeric, cannot fill with median.')
    return dataframe


def fill_na_with_constant(dataframe, columns, constant_value="Undefined"):
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        dataframe[column] = dataframe[column].fillna(constant_value)  # Uzupełnianie braków danych wartością stałą
    return dataframe


def skipping_na_rows(dataframe, columns):
    if isinstance(columns, str):
        columns = [columns]
    return dataframe.dropna(subset=columns)                 # Pomijanie wierszy z brakami danych


def skipping_na_columns(dataframe, columns):
    if isinstance(columns, str):
        columns = [columns]
    return dataframe.drop(columns=columns)                   # Pomijanie kolumn z brakami danych

In [ ]:
# Przykładowe użycie funkcji do uzupełniania braków danych w zbiorze train

train = skipping_na_rows(train, columns='Embarked')   # - usunięcie wierszy z brakami danych w kolumnie 'Embarked'
train = fill_na_with_mean(train, columns='Age')       # - uzupełnienie braków danych w kolumnie 'Age' średnią arytmetyczną
train = fill_na_with_mode(train, columns='Cabin')     # - uzupełnienie braków danych w kolumnie 'Cabin' najczęściej występującą wartością

print(train.isna().sum())

In [ ]:
# Przykładowe użycie funkcji do uzupełniania braków danych w zbiorze test

test = skipping_na_columns(test, columns='Cabin')      # - usunięcie kolumny 'Cabin', ponieważ ma dużo braków danych
test = fill_na_with_median(test, columns='Age')       # - uzupełnienie braków danych w kolumnie 'Age' medianą
test = fill_na_with_constant(test, columns='Fare')   # - uzupełnienie braków danych w kolumnie 'Fare' wartością "Undefined"

print(test.isna().sum())

## Zadanie 4

Przetwarzanie danych kategorialnych: Wykorzystaj kolumny kategorialne (np. płeć, port zaokrętowania) ze zbioru Titanic. Przekształć je na format numeryczny, używając techniki One-Hot Encoding z biblioteki pandas (`pd.get dummies()`) oraz scikit-learn.

In [ ]:
pd.get_dummies(
    train,
    columns=['Sex', 'Embarked'],
    prefix=['Sex', 'Emb'],
    # dummy_na = True, # Dodaje kolumnę dla brakujących wartości
    dtype=int
).head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder


data = train[['Sex', 'Embarked']]

enc = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore',
    categories='auto',
    dtype=int
)

enc.fit(data)

names = enc.get_feature_names_out()
x = enc.transform(data)

transformed_dataframe = pd.DataFrame(x, columns=names)
transformed_dataframe.head()

## Zadanie 5

Skalowanie danych numerycznych: Wybierz kolumny numeryczne ze zbioru Iris (np. długość i szerokość płatka). Zastosuj dwie różne metody skalowania:
 - Standaryzacja (StandardScaler): Przekształć dane tak, aby
miały średnią 0 i odchylenie standardowe 1.
 -  Normalizacja (MinMaxScaler): Przekształć dane tak, aby ich
wartości mieściły się w zakresie od 0 do 1.

Zwizualizuj dane przed i po skalowaniu za pomocą wykresu punktowego.

In [ ]:
iris_df = load_iris(as_frame=True).frame

In [ ]:
def standard_scaler_manual(dataframe, columns):
    new_dataframe = dataframe.copy()
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        mean = dataframe[column].mean()
        std = dataframe[column].std(ddof=0)  # Parametr definiuje rodzaj odchylenia standardowego
        new_dataframe[column] = (dataframe[column] - mean) / std
    return new_dataframe

In [ ]:
normalized_data = standard_scaler_manual(iris_df.copy(), columns=['sepal length (cm)', 'sepal width (cm)'])
normalized_data[['sepal length (cm)', 'sepal width (cm)']].describe()

In [ ]:
plt.scatter(iris_df['sepal length (cm)'], iris_df['sepal width (cm)'])
plt.scatter(normalized_data['sepal length (cm)'], normalized_data['sepal width (cm)'])
plt.xlabel('sepal length (cm)')
plt.ylabel('sepal width (cm)')
plt.title('Standard Scaler Manual')
plt.legend(['Original', 'Standard Scaled'])
plt.grid(True)
plt.show()

In [ ]:
scaler = StandardScaler()
scaler.fit(iris_df[['sepal length (cm)', 'sepal width (cm)']].dropna())
sklearn_scaled = iris_df.copy()
sklearn_scaled[['sepal length (cm)', 'sepal width (cm)']] = scaler.transform(iris_df[['sepal length (cm)', 'sepal width (cm)']])

sklearn_scaled[['sepal length (cm)', 'sepal width (cm)']].describe()

In [ ]:
plt.scatter(iris_df['sepal length (cm)'], iris_df['sepal width (cm)'])
plt.scatter(sklearn_scaled['sepal length (cm)'], sklearn_scaled['sepal width (cm)'])
plt.xlabel('sepal length (cm)')
plt.ylabel('sepal width (cm)')
plt.title('Standard Scaler Sklearn')
plt.legend(['Original', 'Standard Scaled'])
plt.grid(True)
plt.show()

In [ ]:
def min_max_scaler_manual(dataframe, columns):
    new_dataframe = dataframe.copy()
    if isinstance(columns, str):
        columns = [columns]
    for column in columns:
        min_value = dataframe[column].min()
        max_value = dataframe[column].max()
        new_dataframe[column] = (dataframe[column] - min_value) / (max_value - min_value)
    return new_dataframe

In [ ]:
min_max_normalized = min_max_scaler_manual(iris_df.copy(), columns=['sepal length (cm)', 'sepal width (cm)'])
min_max_normalized[['sepal length (cm)', 'sepal width (cm)']].describe()

In [ ]:
plt.scatter(iris_df['sepal length (cm)'], iris_df['sepal width (cm)'])
plt.scatter(min_max_normalized['sepal length (cm)'], min_max_normalized['sepal width (cm)'])
plt.xlabel('sepal length (cm)')
plt.ylabel('sepal width (cm)')
plt.title('Min Max Scaler Manual')
plt.legend(['Original', 'Min Max Scaled'])
plt.grid(True)
plt.show()

In [ ]:
scaler = MinMaxScaler()
scaler.fit(iris_df[['sepal length (cm)', 'sepal width (cm)']].dropna())
min_max_sklearn_scaled = iris_df.copy()
min_max_sklearn_scaled[['sepal length (cm)', 'sepal width (cm)']] = scaler.transform(iris_df[['sepal length (cm)', 'sepal width (cm)']])

min_max_sklearn_scaled[['sepal length (cm)', 'sepal width (cm)']].describe()

In [ ]:
plt.scatter(iris_df['sepal length (cm)'], iris_df['sepal width (cm)'])
plt.scatter(min_max_sklearn_scaled['sepal length (cm)'], min_max_sklearn_scaled['sepal width (cm)'])
plt.xlabel('sepal length (cm)')
plt.ylabel('sepal width (cm)')
plt.title('Min Max Scaler Sklearn')
plt.legend(['Original', 'Min Max Scaled'])
plt.grid(True)
plt.show()

## Zadanie 6

Podział zbioru danych: Dokonaj podziału zbioru Iris na zbiór treningowy (70%) i testowy (30%) za pomocą funkcji train test split
z biblioteki scikit-learn. Sprawdź wymiary powstałych zbiorów.

In [ ]:
from sklearn.model_selection import train_test_split


df = load_iris(as_frame=True).frame

train_x, test_x, train_y, test_y = train_test_split(
    df.drop(columns='target'),
    df['target'],
    test_size=0.3,
    # random_state=7
)

print('train_x shape: ', train_x.shape)
print('train_y shape: ', train_y.shape)
print('test_x shape: ', test_x.shape)
print('test_y shape: ', test_y.shape)

## Zadanie 7

Przygotowanie danych do modelowania: Wybierz dowolny zbiór danych i przeprowadź pełny proces przygotowania danych: wczytanie, obsługa brakujących wartości, przetwarzanie danych kategorialnych i skalowanie danych numerycznych. Zapisz przygotowany zbiór do pliku.

In [ ]:
salary_df = pd.read_csv('Salary_Data.csv')
display(salary_df.head())

In [ ]:
salary_df.info()

In [ ]:
print(salary_df.isna().sum())
salary_df['Age'][salary_df['Age'].isna()]

In [ ]:
salary_df = fill_na_with_mode(salary_df, columns=['Gender', 'Education Level', 'Job Title'])
salary_df = fill_na_with_mean(salary_df, columns=['Age', 'Years of Experience', 'Salary'])

In [ ]:
print(salary_df.isna().sum())

In [ ]:
salary_df = pd.get_dummies(
    salary_df,
    columns=['Gender', 'Education Level', 'Job Title'],
    dtype=int
)

display(salary_df.head())

In [ ]:
salary_df = standard_scaler_manual(salary_df, columns=['Age', 'Years of Experience', 'Salary'])
display(salary_df.head())

In [ ]:
salary_df.to_csv('Salary_Data_Preprocessed.csv')